# Authorized UiO-66 eight-design pilot
User approved ONLY 512 guest relaxations plus 16 registered force continuations,
two T4s, at most 4.2 hours including setup from the conservative recorded start.
No refit, full 64-design expansion, DFT or material-ranking claim.
Run cells individually. Code/poses/notebooks/checkpoint identities frozen in the
private release. STOP file ends owned jobs and preserves raw/failed outputs.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path
release_paths=list(Path('/kaggle/input').rglob('execution_release.json'))
assert len(release_paths)==1, 'Attach the private authorized pilot release'
AUTH=json.loads(release_paths[0].read_text())
bundles=list(Path('/kaggle/input').rglob('UIO66_Sampling_Pilot_v2_authorized.bin'))
assert len(bundles)==1
BUNDLE=bundles[0]
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()==AUTH['bundle_sha256']
REPO=Path('/kaggle/working/MOF_sampling_authorized_portable')
assert not REPO.exists(), 'Do not overwrite a previous extraction'
with zipfile.ZipFile(BUNDLE) as archive:
    for name in archive.namelist():
        assert (REPO/name).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
manifest=json.loads((REPO/'bundle_manifest.json').read_text())['sha256']
assert manifest==AUTH['file_sha256']
for name,digest in manifest.items():
    file=REPO/name
    assert hashlib.sha256(file.read_bytes()).hexdigest()==digest,name
    file.chmod(0o444)
os.chdir(REPO)
ENV=dict(os.environ,PYTHONPATH=str(REPO),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',PYTHONDONTWRITEBYTECODE='1')
ALLOCATION_START_UTC=AUTH['allocation_budget_start_utc']
PRE=Path('/kaggle/working/uio66_sampling_preflight_portable')
assert not PRE.exists()
PRE.mkdir()
gpu_info=subprocess.check_output(['nvidia-smi','--query-gpu=index,name,memory.total,uuid','--format=csv,noheader'],text=True)
assert len(gpu_info.strip().splitlines())==2 and all('T4' in r for r in gpu_info.strip().splitlines())
(PRE/'allocation_observation.json').write_text(json.dumps({'gpu_observed_utc':datetime.now(timezone.utc).isoformat(),
 'budget_start_utc':ALLOCATION_START_UTC,'basis':'conservative timestamp frozen before session start; includes setup',
 'gpu_info':gpu_info,'release_sha256':hashlib.sha256(release_paths[0].read_bytes()).hexdigest()},indent=2)+chr(10))
print('FROZEN RELEASE VERIFIED:',AUTH['bundle_sha256'])
print(gpu_info)
with (PRE/'install.log').open('w') as log:
    remaining=15120-(datetime.now(timezone.utc)-datetime.fromisoformat(ALLOCATION_START_UTC)).total_seconds()
    subprocess.run([sys.executable,'-m','pip','install','-q','fairchem-core==2.23.0','ase==3.26.0'],
        stdout=log,stderr=subprocess.STDOUT,check=True,timeout=max(1,remaining))
subprocess.run([sys.executable,'-m','scripts.run_uio66_sampling_pilot','--output',str(PRE/'dry'/'report.json')],env=ENV,check=True)
print('GPU-environment bundle/512-pose geometry preflight passed; no model loaded yet')


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
plan=json.loads((REPO/'data/design/uio66_sampling_pilot_v2.json').read_text())
CHECKPOINT=Path(hf_hub_download('facebook/UMA','checkpoints/uma-s-1p2p1.pt',
 revision=plan['checkpoint_revision'],token=os.environ['HF_TOKEN']))
digest=hashlib.sha256()
with CHECKPOINT.open('rb') as stream:
    for block in iter(lambda:stream.read(8*1024**2),b''): digest.update(block)
assert digest.hexdigest()==plan['checkpoint_sha256']==AUTH['checkpoint_sha256']
(PRE/'checkpoint_verification.json').write_text(json.dumps({'sha256':digest.hexdigest(),
 'revision':plan['checkpoint_revision'],'verified_utc':datetime.now(timezone.utc).isoformat()},indent=2)+chr(10))
print('PINNED CHECKPOINT VERIFIED:',digest.hexdigest())


In [ ]:
OUT=Path('/kaggle/working/uio66_sampling_authorized')
assert not OUT.exists(), 'New output root required'
controller_log=Path('/kaggle/working/uio66_sampling_controller.log')
with controller_log.open('w') as log:
    CONTROLLER=subprocess.Popen([sys.executable,'-m','scripts.control_uio66_sampling_pilot',
     '--checkpoint',str(CHECKPOINT),'--bundle',str(BUNDLE),'--bundle-sha256',AUTH['bundle_sha256'],
     '--allocation-start-utc',ALLOCATION_START_UTC,'--output',str(OUT)],env=ENV,
     stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print('Controller launched:',CONTROLLER.pid,'| Deadline:',ALLOCATION_START_UTC,'+4.2h')
print('Chemistry stop: write a specific reason to',OUT/'STOP')


In [ ]:
from collections import Counter
print('Controller exit:',CONTROLLER.poll())
if (OUT/'allocation.json').exists():
    allocation=json.loads((OUT/'allocation.json').read_text())
    print('STATUS:',allocation['status'],'| DEADLINE:',allocation['deadline_utc'])
    print('STOP:',allocation.get('stop_reason'))
for i in range(4):
    report=OUT/f'worker{i}'/'adsorption.json'
    if report.exists():
        data=json.loads(report.read_text())
        active=data.get('active_candidate') or {}
        completed=[s for r in data['results'] for s in r.get('starts',[])]+active.get('starts',[])
        print(i,'finished designs',len(data['results']),'active',active.get('id'),
              'guest attempts',len(completed),'statuses',dict(Counter(s['status'] for s in completed)))
print(controller_log.read_text()[-1600:] if controller_log.exists() else '')


In [ ]:
assert CONTROLLER.poll() is not None, 'Keep monitoring; never archive live trajectories as final'
from IPython.display import FileLink, display
for name in ('combined_assessment.json','loose_assessment.json'):
    if (OUT/name).exists():
        result=json.loads((OUT/name).read_text())
        print(name,result['stability'],result['force_check'])
combined=Path('/kaggle/working/UIO66_sampling_authorized_evidence.zip')
assert not combined.exists()
with zipfile.ZipFile(combined,'w',zipfile.ZIP_DEFLATED) as archive:
    for folder in (OUT,PRE):
        for file in sorted(folder.rglob('*')):
            if file.is_file(): archive.write(file,str(file.relative_to('/kaggle/working')))
    archive.write(controller_log,controller_log.name)
print('Evidence SHA256:',hashlib.sha256(combined.read_bytes()).hexdigest(),'bytes',combined.stat().st_size)
display(FileLink(str(combined)))
print('Download evidence and executed notebook; then shut down GPU session. No refit or expansion.')
